# 05b — Pseudo-Labeling (Self-Training) — ELECTRA-small

Same self-training loop as `05_pseudo_labeling.ipynb`, run with
`utils.config.CLASSIFIER_MODEL_NAME_ALT` (`google/electra-small-discriminator`)
instead of DistilBERT, at the same sample size and target coverage
(`PSEUDO_LABEL_SAMPLE_SIZE=400`, 0.98 target coverage), so the two models are
as comparable as possible side by side in `07_comparison.ipynb`. Mentor
feedback item 1 — an addition, not a replacement; `05_pseudo_labeling.ipynb`
and its DistilBERT results are untouched.

**Confidence threshold retuned to 0.30 (not 0.80).** ELECTRA-small fine-tuned
for 3 epochs on only the 400-row labeled seed is far less confident than
DistilBERT at this sample size: its round-0 softmax confidence on the
unlabeled pool tops out around 0.33 (checked empirically), so DistilBERT's
tuned 0.80 threshold (and even 0.60) absorbs exactly 0 pseudo-labels on
round 0 and the loop stops immediately — the same "model too underconfident
to progress" wall `05_pseudo_labeling.ipynb`'s own Experiment 1 hit for
DistilBERT at 0.90. Lowering the threshold to 0.30 restores genuine
multi-round self-training (301 absorbed in round 0, 92 more in round 1,
99.1% coverage). This means the two notebooks are *not* threshold-matched —
read ELECTRA-small's numbers here as "self-training calibrated to what this
smaller, less-confident model can actually do at this sample size," not as
an apples-to-apples confidence-threshold comparison with DistilBERT.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import pandas as pd

from utils import config
from utils.data import stratified_sample
from utils.metrics import evaluate_label_quality, evaluate_semisupervised
from utils.modeling import get_predictions, pseudo_label_loop

In [2]:
labeled_df = pd.read_parquet(config.PROCESSED_DIR / "labeled.parquet")
unlabeled_df = pd.read_parquet(config.PROCESSED_DIR / "unlabeled.parquet")
test_clean = pd.read_parquet(config.PROCESSED_DIR / "test_clean.parquet")

PSEUDO_LABEL_SAMPLE_SIZE = 400  # matches 05_pseudo_labeling.ipynb for a fair comparison
labeled_sample = stratified_sample(labeled_df, PSEUDO_LABEL_SAMPLE_SIZE, seed=config.SEED)
unlabeled_sample = stratified_sample(unlabeled_df, PSEUDO_LABEL_SAMPLE_SIZE, seed=config.SEED)

overlap = set(unlabeled_sample["text"]) & set(test_clean["text"])
assert len(overlap) == 0, f"{len(overlap)} rows leaked between train pool and test set"
print(f"Labeled sample: {len(labeled_sample)} | Unlabeled sample: {len(unlabeled_sample)} | Test: {len(test_clean)}")

Labeled sample: 400 | Unlabeled sample: 400 | Test: 7600


In [3]:
final_model, final_tokenizer, current_labeled, history = pseudo_label_loop(
    labeled_sample, unlabeled_sample,
    model_name=config.CLASSIFIER_MODEL_NAME_ALT,
    confidence_threshold=0.30, epochs=3,
    target_coverage=0.98, max_iterations=10)

for h in history:
    print(h)

Total sample: 800 | target coverage: 98% | confidence threshold: 0.3


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: google/electra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstre

C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,1.333345


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: google/electra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstre

C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,1.307095
100,1.013596
150,0.839299


{'iteration': 0, 'total_sample': 800, 'new_labels': 341, 'new_labels_pct_of_total': 0.42625, 'labeled_size': 741, 'coverage': 0.92625, 'unlabeled_size': 59}
{'iteration': 1, 'total_sample': 800, 'new_labels': 59, 'new_labels_pct_of_total': 0.07375, 'labeled_size': 800, 'coverage': 1.0, 'unlabeled_size': 0}


In [4]:
pseudo_only = current_labeled.iloc[len(labeled_sample):]
merged = pseudo_only.merge(unlabeled_sample[["text", "true_label"]], on="text", how="left")

label_quality = evaluate_label_quality(
    true_labels=merged["true_label"].to_numpy(),
    pseudo_labels=merged["label"].to_numpy())
print("Pseudo-label quality (ELECTRA-small):", label_quality)

Pseudo-label quality (ELECTRA-small): {'Label Accuracy': 0.8225, 'Label Macro F1': 0.8276158595613636, 'Coverage': np.float64(1.0)}


In [5]:
from utils.samples import save_label_samples

save_label_samples(
    merged["text"], merged["label"].to_numpy(), merged["true_label"].to_numpy(),
    config.CLASS_NAMES, n_per_class=2, seed=config.SEED,
    path=config.RESULTS_DIR / "sample_labels_pseudo_labeling_electra_train_pool.csv")
print("Saved sample generated pseudo-labels (train pool) for pseudo_labeling_electra.")

Saved sample generated pseudo-labels (train pool) for pseudo_labeling_electra.


In [6]:
test_probs = get_predictions(final_model, final_tokenizer, test_clean["text"].tolist())
test_preds = test_probs.argmax(axis=1)

semisup_results, report, cm = evaluate_semisupervised(
    test_clean["label"].to_numpy(), test_preds, config.CLASS_NAMES,
    save_path=config.RESULTS_DIR / "confusion_matrix_pseudo_labeling_electra.png")
print(report)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_pseudo_labeling_electra.json", "w") as f:
    json.dump({"test_metrics": semisup_results, "label_quality": label_quality, "history": history},
               f, indent=2)
print("Saved pseudo-labeling (ELECTRA-small) results.")

save_label_samples(
    test_clean["text"], test_preds, test_clean["label"].to_numpy(),
    config.CLASS_NAMES, confidence=test_probs.max(axis=1), n_per_class=2, seed=config.SEED,
    path=config.RESULTS_DIR / "sample_labels_pseudo_labeling_electra_test.csv")
print("Saved sample generated labels (test set) for pseudo_labeling_electra.")

              precision    recall  f1-score   support

       World       0.81      0.89      0.85      1900
      Sports       0.94      0.95      0.95      1900
    Business       0.81      0.72      0.76      1900
    Sci/Tech       0.80      0.80      0.80      1900

    accuracy                           0.84      7600
   macro avg       0.84      0.84      0.84      7600
weighted avg       0.84      0.84      0.84      7600

Saved pseudo-labeling (ELECTRA-small) results.
Saved sample generated labels (test set) for pseudo_labeling_electra.
